<a href="https://colab.research.google.com/github/wjdwogns2873-web/deep-learning-study/blob/main/07_House_Prices.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. 구글 드라이브 연결 (로그인 팝업이 뜨면 확인만 눌러주세요)
from google.colab import drive
drive.mount('/content/drive')

# 2. 구글 드라이브에 저장해둔 열쇠(access_token)를 코랩 보안 폴더로 자동 복사
!mkdir -p ~/.kaggle
!cp /content/drive/MyDrive/Kaggle/access_token ~/.kaggle/
!chmod 600 ~/.kaggle/access_token

# 3. 캐글 API로 MNIST 데이터 초고속 다운로드 및 압축 해제
!pip install -q kaggle

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('house-prices-advanced-regression-techniques')

print("Path to competition files:", path)

In [ ]:
import os

for dirname, _, filenames in os.walk(path):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
import pandas as pd

train_df = pd.read_csv(os.path.join(path, 'train.csv'))
test_df = pd.read_csv(os.path.join(path, 'test.csv'))
# train_df.head()
# test_df.head()

mean_price = train_df['SalePrice'].mean()
print(f"훈련 데이터의 평균 집값: ${mean_price:,.2f}")

print(test_df[['Id', 'GrLivArea', 'TotRmsAbvGrd']].head())

In [ ]:
# 훈련 데이터에서 [면적 1단위(sqft)당 평균 가격]을 계산해봅니다.
# 총 집값을 총 면적으로 나누는 단순한 규칙입니다.
price_per_sqft = train_df['SalePrice'].sum() / train_df['GrLivArea'].sum()
print(f"면적 1단위당 계산된 가격: ${price_per_sqft:.2f}")

# 이제 시험지(test_df)에 있는 집들의 면적에 이 가격을 곱해서 예측값들을 만듭니다.
predictions = []
for area in test_df['GrLivArea']:
    predicted_price = area * price_per_sqft
    predictions.append(predicted_price)

print(len(predictions))

In [ ]:
# 1. 캐글 규격 딕셔너리 생성
submission_data = {
    'Id': test_df['Id'],
    'SalePrice': predictions
}

# 2. 데이터프레임 변환
submission_df = pd.DataFrame(submission_data)

# 3. CSV 파일로 내보내기 (인덱스 제외 필수!)
submission_df.to_csv('house_price_submission.csv', index=False)

In [ ]:
import pandas as pd

train_set = pd.read_csv(os.path.join(path, 'train.csv'))

print(train_set.shape)
print(train_set.columns.tolist())
print(train_set.dtypes)

In [ ]:
import numpy as np

missing = train_set.isnull().sum()
missing = missing[missing > 0]
print("Features with missing values:")
print(missing.sort_values(ascending=False))

cat_cols = train_set.select_dtypes(include=["object"]).columns

print('\nCategorical features:')
print(cat_cols.tolist())

num_cols = train_set.select_dtypes(include=["int64", "float64"]).columns

print("\nNumerical features:")
print(num_cols.tolist())

skewness = train_set[num_cols].skew().sort_values(ascending=False)

print("\nMost skewed numerical features:")
print(skewness.head(10))

In [ ]:
import matplotlib.pyplot as plt

features = ["SalePrice", "OverallQual", "GrLivArea", "GarageCars"]

for feature in features:
    plt.figure(figsize=(6, 4))
    plt.hist(train_set[feature], bins=30)
    plt.title(f"Histogram of {feature}")
    plt.xlabel(feature)
    plt.ylabel("Frequency")
    plt.show()

`plt.show()`를 for문 안에 두어야 하는 이유

matplotlib은 도화지에 그림을 그리는 것입니다.

`plt.figure()`를 호출할 때마다 새 도화지를 한 장씩 꺼냅니다.

코랩 환경이면 눈치껏 분리해서 보여주기도 하지만, 일반적인 파이썬 실행 환경에서
`plt.show()`를 밖으로 빼버리면 그래프가 겹치거나, 마지막 그래프만 노출되는 경우도 있습니다.


In [ ]:
import seaborn as sns

corr = train_set[num_cols].corr()

plt.figure(figsize=(12, 10))

sns.heatmap(
    corr,
    cmap='coolwarm',
    center=0
)

plt.title("Correlation Heatmap")
plt.show()

train_set[num_cols].corr()의 원리

corr()은 통계학의 상관계수를 구하는 함수입니다.

두 변수가 "한쪽이 커질 때 다른 쪽도 같이 커지는가?"를 조사해서 -1~1 숫자로 점수를 매깁니다.

1에 가까울수록(진한 빨간색): "완벽한 패밀리" 하나가 커지면 다른 하나도 무조건 커집니다.(예: 집 평수가 커지면 집값도 커진다.)

-1에 가까울수록(진한 파란색): "완벽한 청개구리" 하나가 커지면 다른 하나는 뚝 떨어집니다.(예: 연식이 오래된 중고차일수록 차 가격은 떨어진다.)

0에 가까울수록(연한 회색/살구색): 아무런 상관이 없는 사이입니다.

cmap='coolwarm', center=0

양수(1)는 따뜻한 색(빨간색), 음수는 차가운 색(파란색)

아무 관계 없는 0점짜리 데이터를 완벽한 무채색(연한 회색/살구색)으로 중심을 잡으라는 뜻입니다.

In [ ]:
target_corr = corr["SalePrice"].sort_values(ascending=False)
print(target_corr.head(15))

이 데이터를 본 고수들의 다음 전처리 전략

1. 샴쌍둥이 제거 고민: GarageCars와 GarageArea는 점수도 비슷하고 의미도 겹치니, 나중에 모델의 부담을 줄이기 위해 둘 중 하나만 쓸까?

2. 새로운 변수 창조: 예를 들어 1층 면적(1stFlrSF)과 지하 면적(TotalBsmtSF)을 더해서 "집의 총 면적(TotalSF)"이라는 새로운 치트키 컬럼을 만들면, 모델이 훨씬 더 예측을 잘하게 됩니다.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

X_train = train_set.drop('SalePrice', axis=1)
y_train = train_set['SalePrice']

num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X_train.select_dtypes(include=["object"]).columns

# 숫자형 컬럼과 글자형 컬럼을 다 다르게 다뤄야 해서 ColumnTransformer를 썼습니다.
# SimpleImputer(strategy="median")
# 숫자 컬럼에 빈칸이 있으면, 그 컬럼의 중간값(Median)을 구해서 빈칸을 자동으로 채워줍니다.
# OneHotEncoder(handle_unknown="ignore")
# 아까 include=["object"]로 뽑아낸 글자 데이터를 모델이 이해할 수 있게 0과 1로 이루어진 숫자로 변환(인코딩)해줍니다.
# "ignore": 나중에 Test에서 처음 보는 이름이 나와도 당황하지 말고 그냥 0으로 처리해 라는 의미입니다.
preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
])

궁금증: OneHotEncoder. 글자 데이터를 모델이 이해할 수 있게 숫자로 변환(인코딩)하는건데, 문자를 어떻게 숫자로 변환하는거고, 그렇게 숫자로 변환해서 활용이 가능한가?

글자를 그냥 숫자로 바꾸는 게 아니라, "새로운 O/X 질문지(컬럼)를 만들어서 0과 1로 대답하게 만드는 기술"입니다.

예를 들어, RoofStype(지붕모양)이라는 문자열 컬럼이 있고, 여기에 Gable, Mansard, Flat이 있다고 하자

그러면 새로운 컬럼을 3개 생성합니다. is_gable, is_mansard, is_flat

Gable이라면 각각 1 0 0이고, Mansard라면 각각 0 1 0이고, Flat이라면 각각 0 0 1이 됩니다.

그 다음 XGBoost(또는 Decision Tree 기반 모델)이 이진트리(Yes or No)를 만들면서 학습을 합니다.

즉, "특정 글자가 들어왔을 때 집값이 오르고 내리는 패턴"을 완벽하게 학습하고 활용할 수 있게 되는 것입니다.


In [ ]:
from sklearn.model_selection import cross_val_score
from xgboost import XGBRegressor
import numpy as np

# 전처리(preprocessor) -> 모델 학습(XGBRegressor) 과정을 하나의 파이프라인으로 묶어버립니다.
# XGBRegressor는 무엇인가?
# 캐글 정형 데이터에서 우승 예측 모델의 80% 이상이 사용하는 강력한 머신러닝 모델입니다.
# 얕은 나무(Decision Tree) 여러개를 이어 붙여가며 앞의 나무가 틀린 오답을 뒤의 나무가 고쳐나가는 방식으로 작동합니다.
pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", XGBRegressor(random_state=0))
])

# cv=5: 데이터를 5등분 합니다.
# 5등분 하는 이유는.. 우연히 특정 데이터만 잘 맞히는 '뽀록'을 방지하고, 모델의 진짜 평균 실력을 공정하게 측정하기 위함입니다.
# RMSE (Root Mean Squared Error): "내가 예측한 집값"과 "실제 집값"의 오차를 계산하는 점수입니다.
# 마이너스를 붙인 이유? 사이킷런 라이브러리는 점수가 높을수록 좋다고 판단하게 설계되어 있습니다.
# 하지만 오차는 낮을수록 좋기 때문에, 억지로 마이너스를 붙여서 음수로 점수를 낮춰서 계산합니다.
scores = -cross_val_score(
    pipeline,
    X_train,
    y_train,
    scoring="neg_root_mean_squared_error",
    cv=5
)

print("CV RMSE:", scores.mean())